In [13]:
%sql
create catalog aws;
use catalog aws ;
create schema test;
use schema test;
create volume v1;

OK

In [0]:
print(dbutils.secrets.listScopes())
print(dbutils.secrets.list(scope='aws'))

[SecretScope(name='aws'), SecretScope(name='azure')]
[SecretMetadata(key='access-key'), SecretMetadata(key='access-secret')]


I'm using manual way to ingest raw data into my uc
1. Using the Boto3 bcoz it's a free edition Databricks 
2. First listing the folders inside the s3 bucket , then using those output displaying the files inside the folders
3. Downloading those files dynamically by creating a mkdir and also downloading the files using the mkdir with files names into my UC 

In [0]:
import boto3

aws_id = dbutils.secrets.get(scope='aws', key='access-key')
aws_secret = dbutils.secrets.get(scope='aws', key='access-secret')

s3 = boto3.client('s3', aws_access_key_id=aws_id, aws_secret_access_key=aws_secret)

In [0]:
import boto3
response = s3.list_objects_v2(
    Bucket='de-prj',
    Delimiter='/'
)
for i in response.get('CommonPrefixes', []):
    folder_name=i['Prefix'].rstrip('/')
    print(f"\n===== Folder: {folder_name} =====")
    folder_path=f'/Volumes/aws/test/v1/{folder_name}'
    dbutils.fs.mkdirs(folder_path)
    print(f'\nCreated the folder {folder_path}')
# print('Checking the ')
    r1=s3.list_objects_v2(
    Bucket='de-prj',
    Prefix=folder_name)
    if 'Contents' in r1:
        for i in r1['Contents']:
            files = i["Key"]
            if files.endswith('/'):
                continue
            file_name = files.split('/')[-1]
            local_path = f'{folder_path}/{file_name}'
            print(f"Downloading {files}")
            s3.download_file(
                'de-prj',
                files,
                local_path
            )
            print(f'File {files} --> {local_path}')
    else:
        print('No Files Founded')


===== Folder: csv_files =====

Created the folder /Volumes/aws/test/v1/csv_files
File csv_files/dim_customers.csv --> /Volumes/aws/test/v1/csv_files/dim_customers.csv
File csv_files/dim_delivery_agents.csv --> /Volumes/aws/test/v1/csv_files/dim_delivery_agents.csv
File csv_files/dim_locations.csv --> /Volumes/aws/test/v1/csv_files/dim_locations.csv
File csv_files/dim_menu_items.csv --> /Volumes/aws/test/v1/csv_files/dim_menu_items.csv
File csv_files/dim_restaurants.csv --> /Volumes/aws/test/v1/csv_files/dim_restaurants.csv

===== Folder: json_files =====

Created the folder /Volumes/aws/test/v1/json_files
File json_files/orders_2024_01.json --> /Volumes/aws/test/v1/json_files/orders_2024_01.json
File json_files/orders_2024_02.json --> /Volumes/aws/test/v1/json_files/orders_2024_02.json
File json_files/orders_2024_03.json --> /Volumes/aws/test/v1/json_files/orders_2024_03.json
File json_files/orders_2024_04.json --> /Volumes/aws/test/v1/json_files/orders_2024_04.json
File json_files/or

### TESTING CELLS 


In [92]:
dbutils.fs.rm('/Volumes/aws/test/v1/csv_files/',recurse=True)
dbutils.fs.rm('/Volumes/aws/test/v1/json_csv/',recurse=True)
dbutils.fs.mkdirs("/Volumes/aws/test/v1/test_folder")

True

In [0]:
files=i["Key"]
file_name=files.split('/')[-1]
print(file_name)
local_path=f'/Volumes/aws/test/v1/test_folder/{file_name}'
s3.download_file('de-prj',files,local_path)
print(f'{files} --> {local_path}')

orders_2024_12.json
json_csv/orders_2024_12.json --> /Volumes/aws/test/v1/test_folder/orders_2024_12.json


In [0]:
print('Displaying the folders and files stored in the s3 bucket')
import boto3
response = s3.list_objects_v2(
    Bucket='de-prj',
    Delimiter='/'
)
for i in response.get('CommonPrefixes', []):
    folder_name=i['Prefix']
    print(f"\n===== Folder: {folder_name} =====")
# print('Checking the ')
    r1=s3.list_objects_v2(
    Bucket='de-prj',
    Prefix=folder_name)
    if 'Contents' in r1:
        for i in r1['Contents']:
            print(i['Key'])
    else:
        print('No files founded')



===== Folder: csv_files/ =====
csv_files/
csv_files/dim_customers.csv
csv_files/dim_delivery_agents.csv
csv_files/dim_locations.csv
csv_files/dim_menu_items.csv
csv_files/dim_restaurants.csv

===== Folder: json_csv/ =====
json_csv/
json_csv/orders_2024_01.json
json_csv/orders_2024_02.json
json_csv/orders_2024_03.json
json_csv/orders_2024_04.json
json_csv/orders_2024_05.json
json_csv/orders_2024_06.json
json_csv/orders_2024_07.json
json_csv/orders_2024_08.json
json_csv/orders_2024_09.json
json_csv/orders_2024_10.json
json_csv/orders_2024_11.json
json_csv/orders_2024_12.json
